In [1]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


# **Download Data**

In [ ]:
!cp ./gdrive/MyDrive/bracs/train.zip ./train.zip
!unzip train.zip
!rm train.zip
!rm -r ./train/1
!rm -r ./train/2
!rm -r ./train/4
!rm -r ./train/6

In [ ]:
!cp ./gdrive/MyDrive/bracs/bracs_in_test.zip ./bracs_in_test.zip
!unzip bracs_in_test.zip
!rm bracs_in_test.zip
!rm -r ./test/1
!rm -r ./test/2
!rm -r ./test/4
!rm -r ./test/6

# **Download Code**

In [ ]:
!pip install lightly

# **Ready for Train**

In [5]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pytorch_lightning as pl
import torch
import torch.nn as nn
import torchvision
from PIL import Image
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import normalize
from tqdm import tqdm
import json

from lightly.data import LightlyDataset
from lightly.transforms import  utils
from lightly.transforms.vicreg_transform import VICRegTransform

In [6]:
num_workers = 2
batch_size = 32
seed = 1
max_epochs = 2
input_size = 224

In [ ]:
pl.seed_everything(seed)

In [8]:
path_to_data_train = "./train"
path_to_data_test = "./test"

In [9]:
transform = VICRegTransform(input_size=input_size)

test_transform = torchvision.transforms.Compose(
    [
        torchvision.transforms.Resize((input_size, input_size)),
        torchvision.transforms.ToTensor(),
    ]
)

dataset_train_vicreg = LightlyDataset(input_dir=path_to_data_train, transform=transform)

dataset_test = LightlyDataset(input_dir=path_to_data_test, transform=test_transform)

dataloader_train_vicreg = torch.utils.data.DataLoader(
    dataset_train_vicreg,
    batch_size=batch_size,
    shuffle=True,
    drop_last=True,
    num_workers=num_workers,
)

dataloader_test = torch.utils.data.DataLoader(
    dataset_test,
    batch_size=batch_size,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers,
)

# **Load Official Checkpoints**

In [ ]:
resnet50 = torch.hub.load('facebookresearch/vicreg:main', 'resnet50')
!wget https://dl.fbaipublicfiles.com/vicreg/resnet50.pth

# resnet50x2 = torch.hub.load('facebookresearch/vicreg:main', 'resnet50x2')
# !wget https://dl.fbaipublicfiles.com/vicreg/resnet50x2.pth

# resnet200x2 = torch.hub.load('facebookresearch/vicreg:main', 'resnet200x2')
# !wget https://dl.fbaipublicfiles.com/vicreg/resnet200x2.pth

In [12]:
ckp = torch.load("./resnet50.pth")
resnet50.load_state_dict(ckp)

<All keys matched successfully>

In [13]:
from lightly.loss.vicreg_loss import VICRegLoss
from lightly.models.modules.heads import VICRegProjectionHead


class VICRegModel(pl.LightningModule):
    def __init__(self):
        super().__init__()

        self.backbone = resnet50

        hidden_dim = 2048
        self.projection_head = VICRegProjectionHead(
            input_dim=hidden_dim,
            hidden_dim=2048,
            output_dim=2048,
            num_layers=2,
        )
        self.criterion = VICRegLoss()

    def forward(self, x):
        h = self.backbone(x).flatten(start_dim=1)
        z = self.projection_head(h)
        return z

    def training_step(self, batch, batch_idx):
        (x0, x1), _, _ = batch
        z0 = self.forward(x0)
        z1 = self.forward(x1)
        loss = self.criterion(z0, z1)
        self.log("train_loss_ssl", loss)
        return loss

    def configure_optimizers(self):
        optim = torch.optim.SGD(
            self.parameters(), lr=6e-2, momentum=0.9, weight_decay=5e-4
        )
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optim, max_epochs)
        return [optim], [scheduler]

In [14]:
model = VICRegModel()

if you have a checkpoint to load, give the path here

In [ ]:
# checkpoint_path = "path/to/checkpoint"
# ckpt = torch.load(checkpoint_path)


# model.load_state_dict(ckpt['state_dict'])

In [16]:
trainer = pl.Trainer(max_epochs=max_epochs, devices=1, accelerator="gpu" if torch.cuda.is_available() else "cpu")
trainer.fit(model, dataloader_train_vicreg)

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:IPU available: False, using: 0 IPUs
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name            | Type                 | Params
---------------------------------------------------------
0 | backbone        | ResNet               | 23.5 M
1 | projection_head | VICRegProjectionHead | 8.4 M 
2 | criterion       | VICRegLoss           | 0     
---------------------------------------------------------
31.9 M    Trainable params
0         Non-trainable params
31.9 M    Total params
127.611   Total estimated model params size (MB)


Training: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=2` reached.


# **Get Embedding**

In [18]:
new_model = VICRegModel()
checkpoint_path = "./lightning_logs/version_0/checkpoints/epoch=1-step=1858.ckpt"
ckpt = torch.load(checkpoint_path)
new_model.load_state_dict(ckpt['state_dict'])

if torch.cuda.is_available():
  new_model.to(torch.device('cuda'))

In [19]:
def generate_embeddings(model, dataloader):
    """Generates representations for all images in the dataloader with
    the given model
    """

    embeddings = []
    targets = []
    filenames = []
    with torch.no_grad():
        for img, target, fnames in tqdm(dataloader):
            img = img.to(model.device)
            emb = model.backbone(img).flatten(start_dim=1)
            embeddings.append(emb)
            targets.append(target)
            filenames.extend(fnames)

    embeddings = torch.cat(embeddings, 0)
    targets = torch.cat(targets, 0)
    return embeddings, targets, filenames


model.eval()
embeddings, targets, filenames = generate_embeddings(new_model, dataloader_test)

100%|██████████| 94/94 [00:10<00:00,  8.81it/s]


In [20]:
print(embeddings.shape)
print(targets.shape)
print(len(filenames))

torch.Size([3000, 2048])
torch.Size([3000])
3000


In [21]:
torch.save(embeddings, "./embeddings.pth")
torch.save(targets, "./labels.pth")
all_file_paths = {"file_names": filenames}
with open("./file_names.json", 'w') as f2:
  json.dump(all_file_paths, f2)